# 시도 가구소득 보조변수 전처리

전국 255개 시군구 소비를 비교할 때 시도별 구매력 차이를 통제하기 위한 보조변수입니다. 이 자료는 **시도 단위**이므로 시군구 개별 소득으로 해석하지 않습니다.

In [1]:
from pathlib import Path
import numpy as np
import polars as pl

ROOT = Path("..") if (Path("..") / "ABP_CONTEST_DATA.csv").exists() else Path(".")
RAW_PATH = ROOT / "data" / "external" / "raw" / "kosis_household_income_sido_2025.json"
OUTPUT_PATH = ROOT / "data" / "external" / "processed" / "kosis_household_income_sido_2025.csv"
ABP_PATH = ROOT / "ABP_CONTEST_DATA.csv"

assert RAW_PATH.exists(), f"원본 파일이 없습니다: {RAW_PATH}"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 출처와 해석

원본은 KOSIS e-지방지표의 **가구소득**이며, 2025년 연간값·단위는 만원입니다. 2026년 상반기 BC카드 소비보다 한 시점 앞선 구조변수이므로, 동시점 개인소득이나 인과효과로 주장하지 않습니다.

- 출처: [KOSIS e-지방지표 가구소득](https://kosis.kr/visual/eRegionIndex/eRegionWhole.do)
- 사용: `log(시도 가구소득)`을 수요모형의 보조 설명변수로 사용
- 금지: 시군구별 소득, 개인의 소득, 또는 카드 소비의 원인으로 해석

In [2]:
income_raw = pl.read_json(RAW_PATH)
assert {"data"} <= set(income_raw.columns)

records = income_raw.explode("data").unnest("data")
latest = records.filter((pl.col("wrtPnttm") == "2025") & (pl.col("regionCd") != "00"))
assert latest.height == 17
assert latest.select("regionNm").n_unique() == 17
latest.select(["regionCd", "regionNm", "wrtPnttm", "unit", "vl"]).sort("regionCd")

/tmp/ipykernel_2556058/1139701229.py:4: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  records = income_raw.explode("data").unnest("data")


regionCd,regionNm,wrtPnttm,unit,vl
str,str,str,str,f64
"""11""","""서울""","""2025""","""만원""",8163.959683
"""21""","""부산""","""2025""","""만원""",6349.487131
"""22""","""대구""","""2025""","""만원""",6537.39349
"""23""","""인천""","""2025""","""만원""",6335.336858
"""24""","""광주""","""2025""","""만원""",6708.348831
…,…,…,…,…
"""35""","""전북""","""2025""","""만원""",6587.183486
"""36""","""전남""","""2025""","""만원""",6764.411509
"""37""","""경북""","""2025""","""만원""",6490.024616


## 시도명 결합

공모전 원본은 법정 명칭을 사용하고 KOSIS는 짧은 명칭을 사용한다. 명시적 교차표로만 변환하며, 결합 실패를 평균값으로 대체하지 않는다.

In [3]:
sido_name_map = pl.DataFrame(
    {
        "regionNm": [
            "서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종", "경기",
            "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주",
        ],
        "SIDO_NM": [
            "서울특별시", "부산광역시", "대구광역시", "인천광역시", "광주광역시", "대전광역시",
            "울산광역시", "세종특별자치시", "경기도", "강원특별자치도", "충청북도", "충청남도",
            "전북특별자치도", "전라남도", "경상북도", "경상남도", "제주특별자치도",
        ],
    }
)

income_sido = (
    latest.join(sido_name_map, on="regionNm", how="left", validate="1:1")
    .select(
        "SIDO_NM",
        pl.lit(2025).alias("INCOME_YEAR"),
        pl.col("vl").cast(pl.Float64).alias("HOUSEHOLD_INCOME_10K_KRW"),
    )
    .with_columns(
        (
            (pl.col("HOUSEHOLD_INCOME_10K_KRW").log() - pl.col("HOUSEHOLD_INCOME_10K_KRW").log().mean())
            / pl.col("HOUSEHOLD_INCOME_10K_KRW").log().std()
        ).alias("HOUSEHOLD_INCOME_LOG_Z")
    )
    .sort("SIDO_NM")
)
assert income_sido.select("SIDO_NM").drop_nulls().n_unique() == 17
assert income_sido.select(pl.col("HOUSEHOLD_INCOME_10K_KRW").min()).item() > 0

abp_sidos = pl.scan_csv(ABP_PATH).select("SIDO_NM").unique().collect()
unmatched_sidos = abp_sidos.join(income_sido.select("SIDO_NM"), on="SIDO_NM", how="anti")
assert unmatched_sidos.is_empty(), unmatched_sidos

income_sido.write_csv(OUTPUT_PATH)
print(f"시도 가구소득 {income_sido.height}행 저장: {OUTPUT_PATH}")
income_sido

시도 가구소득 17행 저장: ../data/external/processed/kosis_household_income_sido_2025.csv


SIDO_NM,INCOME_YEAR,HOUSEHOLD_INCOME_10K_KRW,HOUSEHOLD_INCOME_LOG_Z
str,i32,f64,f64
"""강원특별자치도""",2025,6478.93419,-0.73811
"""경기도""",2025,8487.677075,1.495825
"""경상남도""",2025,6392.408701,-0.849327
"""경상북도""",2025,6490.024616,-0.723963
"""광주광역시""",2025,6708.348831,-0.450271
…,…,…,…
"""전라남도""",2025,6764.411509,-0.381427
"""전북특별자치도""",2025,6587.183486,-0.601044
"""제주특별자치도""",2025,6854.801225,-0.271624


## 모형 투입

시군구 `i`가 속한 시도 `s(i)`에 대해 `q_s = HOUSEHOLD_INCOME_LOG_Z`를 다음처럼 넣는다.

\[
\log\mu_{igb}=\log E_{ig}+\alpha_b+\gamma_{bg}+u_i+v_{ib}+\beta_b^{(q)}q_{s(i)}.
\]

`q_s`는 17개 시도 사이의 구매력 차이를 설명하며, 같은 시도 안의 시군구 차이는 지역 랜덤효과 `u_i`, `v_ib`가 설명한다. `beta_b^(q)`에는 약한 정규 prior를 둔다. 관측기간이 6개월뿐이므로 소득 효과를 인과효과로 해석하지 않는다.

## 소비와의 사전 적합성 확인

이 검사는 소득이 2026년 소비를 **설명하는지**가 아니라, 모델 입력으로서 변동성과 결측이 충분한지를 확인한다. BC카드 원본에는 관측되지 않은 셀의 생성규칙이 알려져 있지 않으므로, 아래 상관은 관측된 양(+) 거래의 기술통계이며 인과효과 검정이 아니다.

In [4]:
POPULATION_PATH = ROOT / "data" / "external" / "processed" / "population_sgg_age_sex_202606.csv"
assert POPULATION_PATH.exists()

domestic_consumption = (
    pl.scan_csv(ABP_PATH)
    .filter(pl.col("GENDER_CD").is_in(["1", "2"]))
    .group_by("SIDO_NM")
    .agg(
        pl.col("cnt").sum().alias("OBSERVED_TRANSACTION_COUNT"),
        pl.col("amt").sum().alias("OBSERVED_AMOUNT_KRW"),
    )
    .collect()
)
population_sido = (
    pl.scan_csv(POPULATION_PATH)
    .group_by("SIDO_NM")
    .agg(pl.col("POPULATION").sum().alias("DOMESTIC_RESIDENT_POPULATION"))
    .collect()
)

income_model_check = (
    income_sido.join(domestic_consumption, on="SIDO_NM", how="inner", validate="1:1")
    .join(population_sido, on="SIDO_NM", how="inner", validate="1:1")
    .with_columns(
        (pl.col("OBSERVED_TRANSACTION_COUNT") / pl.col("DOMESTIC_RESIDENT_POPULATION")).log().alias("LOG_OBSERVED_TX_PER_RESIDENT"),
        (pl.col("OBSERVED_AMOUNT_KRW") / pl.col("DOMESTIC_RESIDENT_POPULATION")).log().alias("LOG_OBSERVED_AMOUNT_PER_RESIDENT"),
        (pl.col("OBSERVED_AMOUNT_KRW") / pl.col("OBSERVED_TRANSACTION_COUNT")).log().alias("LOG_OBSERVED_TICKET"),
    )
    .sort("SIDO_NM")
)
assert income_model_check.height == 17
assert income_model_check.null_count().select(pl.sum_horizontal(pl.all())).item() == 0

income_z = income_model_check["HOUSEHOLD_INCOME_LOG_Z"].to_numpy()
def correlations(column: str) -> tuple[float, float]:
    values = income_model_check[column].to_numpy()
    return (
        float(np.corrcoef(income_z, values)[0, 1]),
        float(np.corrcoef(income_model_check["HOUSEHOLD_INCOME_LOG_Z"].rank(method="average").to_numpy(), income_model_check[column].rank(method="average").to_numpy())[0, 1]),
    )

for label, column in [
    ("거래건수/거주인구", "LOG_OBSERVED_TX_PER_RESIDENT"),
    ("결제금액/거주인구", "LOG_OBSERVED_AMOUNT_PER_RESIDENT"),
    ("객단가", "LOG_OBSERVED_TICKET"),
]:
    pearson_r, spearman_rho = correlations(column)
    print(f"소득–관측 {label} Pearson r: {pearson_r:.3f}, Spearman rho: {spearman_rho:.3f}")

print("판정: 결측 없이 결합 가능. 다만 n=17이므로 계수의 방향·크기는 MCMC와 leave-one-SIDO-out 검증 뒤에만 채택합니다.")
income_model_check


소득–관측 거래건수/거주인구 Pearson r: -0.381, Spearman rho: -0.324
소득–관측 결제금액/거주인구 Pearson r: -0.530, Spearman rho: -0.358
소득–관측 객단가 Pearson r: -0.413, Spearman rho: -0.265
판정: 결측 없이 결합 가능. 다만 n=17이므로 계수의 방향·크기는 MCMC와 leave-one-SIDO-out 검증 뒤에만 채택합니다.


SIDO_NM,INCOME_YEAR,HOUSEHOLD_INCOME_10K_KRW,HOUSEHOLD_INCOME_LOG_Z,OBSERVED_TRANSACTION_COUNT,OBSERVED_AMOUNT_KRW,DOMESTIC_RESIDENT_POPULATION,LOG_OBSERVED_TX_PER_RESIDENT,LOG_OBSERVED_AMOUNT_PER_RESIDENT,LOG_OBSERVED_TICKET
str,i32,f64,f64,i64,i64,i64,f64,f64,f64
"""강원특별자치도""",2025,6478.93419,-0.73811,20687033,459742960000,1507217,2.619242,12.628158,10.008916
"""경기도""",2025,8487.677075,1.495825,187058568,3505496210000,13761783,2.609526,12.447947,9.838421
"""경상남도""",2025,6392.408701,-0.849327,46744390,948595220000,3195351,2.682997,12.601041,9.918043
"""경상북도""",2025,6490.024616,-0.723963,36420901,728234560000,2495919,2.680486,12.583721,9.903236
"""광주광역시""",2025,6708.348831,-0.450271,27247507,492730020000,1385460,2.97893,12.781684,9.802755
…,…,…,…,…,…,…,…,…,…
"""전라남도""",2025,6764.411509,-0.381427,21678801,503158690000,1773646,2.503298,12.555624,10.052326
"""전북특별자치도""",2025,6587.183486,-0.601044,19487725,392130730000,1718633,2.428256,12.337821,9.909566
"""제주특별자치도""",2025,6854.801225,-0.271624,13417680,297597530000,662792,3.007867,13.014791,10.006924


## 모델 채택 규칙

- 소득 변수는 `q_s` 하나의 약한 정규 prior 계수로만 둔다.
- 시도 랜덤절편을 별도로 추가하지 않는다. 17개 시도에서 `q_s`와 시도 절편을 함께 넣으면 식별성이 약해진다.
- `소득 포함`과 `소득 제외` 모형을 시도 단위 leave-one-out 예측과 PSIS-LOO로 비교한다. 예측 성능이 개선되지 않거나 계수의 90% 신용구간이 넓으면 최종 제출 모형에서 뺀다.
- 이 변수는 2025년 시도 가구소득이므로 개인소득·시군구 소득·인과효과라는 표현을 쓰지 않는다.